# ERA5Land data downloader example

This notebook focuses on accessing and downloading ERA5-Land reanalysis data, a high-resolution dataset provided by the Copernicus Climate Data Store (CDS). ERA5-Land offers hourly estimates of land surface variables from 1981 to present, with a spatial resolution of ~9 km. 

There are two possible datasets of ERA5Land: the raw hourly values, or the aggregated dataset:

- Hourly data: https://cds.climate.copernicus.eu/datasets/reanalysis-era5-land
- Post-processed daily aggregated values: https://cds.climate.copernicus.eu/datasets/derived-era5-land-daily-statistics

## To download hourly data

In [ ]:
import cdsapi
from datetime import datetime, timedelta
import os

from src.config import ERA5_API_KEY

url = "https://cds.climate.copernicus.eu/api"
general_directory = ".../..."

dataset = "reanalysis-era5-land" # 

years = ["2008", "2009", "2010", "2011", "2012", "2013", "2014", "2015", "2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023", "2024"]
months = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]

# Function to generate day lists
def generate_day_lists_by_month(year, month):
    # Days in the month
    days_in_month = []
    current_date = datetime(int(year), int(month), 1)
    while current_date.month == int(month):
        days_in_month.append(current_date.strftime('%d'))
        current_date += timedelta(days=1)
    return days_in_month

# Download data peninsula and baleares
for year in years:
    for month in months:
        directory = general_directory + '/2m_temperature_hourly/'
        target = directory + '2mtemp_hourly_' + year + "_" + month + '.nc' # File name
        
        # Check if the file already exists
        if os.path.exists(target):
            print(f"File already exists: {target}. Skipping download.")
            continue
        
        request = {
            "variable": [
                "2m_temperature"
            ],
            "year": year,
            "month": month,
            "day": generate_day_lists_by_month(year, month),
            "time": [
                "00:00", "01:00", "02:00",
                "03:00", "04:00", "05:00",
                "06:00", "07:00", "08:00",
                "09:00", "10:00", "11:00",
                "12:00", "13:00", "14:00",
                "15:00", "16:00", "17:00",
                "18:00", "19:00", "20:00",
                "21:00", "22:00", "23:00"
            ],
            "data_format": "netcdf",
            "download_format": "unarchived",
            "area": [43.99, -9.85, 34.88, 4.59] # Peninsula y baleares
        }

        try:
            client = cdsapi.Client(url=url, key=ERA5_API_KEY)
            client.retrieve(dataset, request, target)
        except:
            print(f"{target} could not be downloaded!")

## To download aggregates (mean, min, max ...)

In [ ]:
import cdsapi
import os

url = "https://cds.climate.copernicus.eu/api"

dataset = "derived-era5-land-daily-statistics"

years = ["2008", "2009", "2010", "2011", "2012", "2013", "2014", "2015", "2016", "2017", "2018", "2019", "2020", "2021", "2022", "2023", "2024"]
months = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10", "11", "12"]

# Descargar datos peninsula y baleares
for year in years:
    for month in months:
        directory = general_directory + '/2m_temperature/'
        target = directory + 'mean_2mtemp_' + year + "_" + month + '.nc' # File name
        
        # Check if the file already exists
        if os.path.exists(target):
            print(f"File already exists: {target}. Skipping download.")
            continue

        request = {
            "variable": [
                "2m_temperature"
            ],
            "year": year,
            "month": month,
            "day": generate_day_lists_by_month(year, month),
            "daily_statistic": "daily_mean",
            "time_zone": "utc+01:00",
            "frequency": "1_hourly",
            "area": [43.99, -9.85, 34.88, 4.59] # Peninsula y baleares
        }

        client = cdsapi.Client(url=url, key=ERA5_API_KEY)
        client.retrieve(dataset, request, target)

